# Medical Data Validator — Quickstart Notebook

This notebook walks a researcher or data steward through the full lifecycle:
1. Validate a dataset
2. Inspect compliance results (HIPAA, GDPR, FDA, FHIR R4, SNOMED CT)
3. Anonymize PHI columns
4. Export a PDF/CSV report
5. Register the dataset and record the run in the audit trail

**Requirements:** `pip install medical-data-validator reportlab`

In [ ]:
import pandas as pd
from medical_data_validator.core import MedicalDataValidator

## 1. Create a sample dataset

Replace this with `pd.read_csv('your_file.csv')` for real data.

In [ ]:
df = pd.DataFrame({
    'patient_id':   ['P001', 'P002', 'P003', 'P004'],
    'patient_name': ['Alice Smith', 'Bob Jones', 'Carol White', 'Dave Brown'],
    'age':          [34, 56, 29, 72],
    'ssn':          ['123-45-6789', None, '987-65-4321', None],
    'email':        ['alice@example.com', 'bob@hospital.org', None, 'dave@clinic.net'],
    'diagnosis':    ['Flu', 'Hypertension', 'Asthma', 'Diabetes'],
    'icd10_code':   ['J11.1', 'I10', 'J45.909', 'E11.9'],
    'visit_date':   ['2024-01-15', '2024-02-20', '2024-03-10', '2024-04-05'],
})
df

## 2. Validate the dataset

`MedicalDataValidator` runs all enabled checks:
- Schema & data-quality rules
- PHI/PII detection
- HIPAA, GDPR, FDA 21 CFR Part 11 compliance scoring

In [ ]:
validator = MedicalDataValidator(
    enable_compliance=True,
    enable_analytics=True,
    enable_monitoring=False,   # disable background thread in notebooks
)

from medical_data_validator.validators import PHIDetector, DataQualityChecker
validator.add_rule(PHIDetector())
validator.add_rule(DataQualityChecker())

result = validator.validate(df)
print(f'Valid: {result.is_valid}')
print(f'Issues: {len(result.issues)} (errors={result.summary.get("error_count",0)}, warnings={result.summary.get("warning_count",0)})')

In [ ]:
# Inspect individual issues
for issue in result.issues[:10]:
    print(f'[{issue.severity.upper()}] {issue.rule_name}: {issue.message}')

## 3. Compliance report

The compliance engine scores the dataset against HIPAA, GDPR, and FDA.

In [ ]:
compliance = result.summary.get('compliance_report', {})
standards  = compliance.get('standards', {})

rows = []
for std, data in standards.items():
    if isinstance(data, dict) and 'score' in data:
        rows.append({
            'Standard':   std.upper(),
            'Score':      round(data.get('score', 0), 1),
            'Risk Level': data.get('risk_level', 'N/A'),
            'Compliant':  '✅' if data.get('compliant') else '❌',
            'Violations': data.get('violations_count', len(data.get('violations', []))),
        })

pd.DataFrame(rows).set_index('Standard')

## 4. FHIR R4 and SNOMED CT validation

Register the built-in plugins for interoperability checks.

In [ ]:
from medical_data_validator.plugins import load_compliance_plugins
from medical_data_validator.compliance import ComplianceEngine

engine = load_compliance_plugins()   # FHIR R4 + SNOMED CT auto-loaded
fhir_report = engine.comprehensive_compliance_validation(df)

for std_name in ('fhir_r4', 'snomed_ct'):
    std = fhir_report['standards'].get(std_name, {})
    print(f'{std_name}: score={std.get("score","N/A")}, violations={std.get("violations_count",0)}')

## 5. Anonymize PHI columns

Three strategies:
- `hipaa_safe_harbor` — replace names/IDs/dates with safe surrogates
- `hash` — SHA-256 hash of each value
- `mask` — partial masking (`***`)

In [ ]:
anon_df = validator.anonymize(
    df,
    columns=['patient_name', 'ssn', 'email'],
    method='hipaa_safe_harbor',
)
anon_df[['patient_name', 'ssn', 'email', 'age', 'diagnosis']]

## 6. Export reports

In [ ]:
from medical_data_validator.reports import generate_csv_report, generate_pdf_report

result_dict = result.to_dict()

# CSV export
csv_text = generate_csv_report(result_dict)
with open('/tmp/validation_report.csv', 'w') as f:
    f.write(csv_text)
print('CSV saved to /tmp/validation_report.csv')

# PDF export
pdf_bytes = generate_pdf_report(result_dict)
with open('/tmp/validation_report.pdf', 'wb') as f:
    f.write(pdf_bytes)
print(f'PDF saved to /tmp/validation_report.pdf ({len(pdf_bytes):,} bytes)')

## 7. Dataset registry and audit trail

In [ ]:
import os, tempfile
import medical_data_validator.registry as registry
import medical_data_validator.audit as audit

# Use a temp DB so this notebook doesn't affect the system audit log
tmp = tempfile.NamedTemporaryFile(suffix='.db', delete=False)
tmp.close()
registry.REGISTRY_DB_PATH = tmp.name
registry._conn = None
audit.AUDIT_DB_PATH = tmp.name
audit._conn = None

# Register the dataset
ds = registry.register_dataset(
    'trial_cohort_2024',
    tenant='my_institute',
    description='Phase II trial cohort data',
    tags=['clinical_trial', 'phase_ii'],
)
print('Dataset registered:', ds['id'])

# Record this validation run
run_id = registry.record_run(
    ds['id'],
    is_valid=result.is_valid,
    error_count=len(result.get_issues_by_severity('error')),
    warning_count=len(result.get_issues_by_severity('warning')),
)
print('Run recorded:', run_id)

# Query history
history = registry.get_run_history(ds['id'])
pd.DataFrame(history)[['id', 'run_at', 'is_valid', 'error_count', 'warning_count']]

## 8. Async job submission (optional)

For large datasets, submit a validation job and poll for results.

In [ ]:
import time, tempfile
import medical_data_validator.jobs as jobs

tmp_jobs = tempfile.NamedTemporaryFile(suffix='.db', delete=False)
tmp_jobs.close()
jobs.JOBS_DB_PATH = tmp_jobs.name
jobs._conn = None
jobs._worker_started = False

job_id = jobs.submit_job(
    'validate',
    {'data': df.to_dict(orient='list'), 'enable_compliance': False},
)
print('Job submitted:', job_id)

# Poll until done
for _ in range(60):
    job = jobs.get_job(job_id)
    if job['status'] not in ('pending', 'running'):
        break
    time.sleep(0.1)

print(f'Status: {job["status"]}')
if job['status'] == 'completed':
    print(f'is_valid: {job["result"]["is_valid"]}, issues: {job["result"]["total_issues"]}')